# Stratified LD Score Regression
This notebook implements the pipeline of [S-LDSC](https://www.nature.com/articles/ng.3404) for LD score and functional enrichment analysis.

**Important: the S-LDSC implementation comes from the [polyfun](https://github.com/omerwe/polyfun/tree/master) package, not the original LDSC from `bulik/ldsc` GitHub repo.**

The purpose of this pipeline is to use LD Score Regression (LDSC) to analyze the heritability and enrichment of genome annotations across GWAS traits. By integrating genome annotation files and GWAS summary statistics, this pipeline allows single tau analysis (individual annotation contributions) and joint tau analysis (independent contributions of multiple annotations after removing shared effects).

The pipeline is developed to integrate GWAS summary statistics data, annotation data, and LD reference panel data to compute functional enrichment for each of the epigenomic annotations that the user provides using the S-LDSC model. We will first start off with an introduction, instructions to set up, and the minimal working examples. Then the workflow code that can be run using SoS on any data will be at the end.

## A brief review on Stratified LD score regression

Here I briefly review LD Score Regression and what it is used for. For more in-depth information on LD Score Regression please read the following three papers:

1. "LD Score regression distinguishes confounding from polygenicity in genome-wide association studies" by Bulik-Sullivan et al. (2015)

2. "Partitioning heritability by functional annotation using genome-wide association summary statistics" by Finucane et al. (2015)

3. "Linkage disequilibrium-dependent architecture of human complex traits shows action of negative selection" by Gazal et al. (2017)

As stated in Bulik-Sullivan et al. (2015), confounding factors and polygenic effects can both inflate test statistics, and most existing methods cannot distinguish inflation from confounding bias from a true polygenic signal. LD Score Regression (LDSC) identifies the relative contribution of confounding versus polygenicity directly from GWAS summary statistics.

The approach regresses the relationship between Linkage Disequilibrium (LD) scores and test statistics of SNPs from the GWAS summary statistics. Variants in LD with a "causal" variant show an elevation in test statistics in association analysis proportional to their LD (measured by $r^2$) with the causal variant within a certain window size (e.g. 1 cM, 1 kb). In contrast, inflation from confounders such as population stratification arising purely from genetic drift will not correlate with LD. For a polygenic trait, SNPs with a high LD score will have more significant $\chi^2$ statistics on average than SNPs with a low LD score. Thus, if we regress the $\chi^2$ statistics from GWAS against LD Score, the intercept minus one is an estimator of the mean contribution of confounding bias to the inflation in the test statistics. The regression model is known as LD Score regression.

### LDSC model

Under a polygenic assumption, in which effect sizes for variants are drawn independently from distributions with variance proportional to $1/(p(1-p))$ where $p$ is the minor allele frequency (MAF), the expected $\chi^2$ statistic of variant $j$ is:

$$E[\chi^2_j \mid \ell_j] \;=\; \frac{N\,h^2\,\ell_j}{M} \;+\; N a \;+\; 1 \quad (1)$$

where $N$ is the sample size; $M$ is the number of SNPs, so that $h^2/M$ is the average heritability per SNP; $a$ measures the contribution of confounding biases such as cryptic relatedness and population stratification; and $\ell_j = \sum_k r^2_{jk}$ is the LD Score of variant $j$, which measures the amount of genetic variation tagged by $j$. A full derivation is given in the Supplementary Note of Bulik-Sullivan et al. (2015); an alternative derivation appears in the Supplementary Note of Zhu and Stephens (2017) AoAS.

Equation (1) shows that LD Score regression can compute SNP-based heritability for a phenotype from GWAS summary statistics alone, without requiring individual-level genotype data as REML and related methods do.

### Stratified LDSC

Heritability is the proportion of phenotypic variation that is due to variation in genetic values, and it can also be partitioned over disjoint or overlapping categories of SNPs.

Stratified LD Score Regression (S-LDSC) partitions heritability by leveraging both LD-score information and SNPs that have not reached genome-wide significance. S-LDSC exploits the fact that the $\chi^2$ statistic for a given SNP reflects the cumulative effects of all SNPs tagged by it: in regions of high LD, the focal SNP captures the contribution of a group of nearby SNPs.

S-LDSC declares an annotation enriched for heritability if SNPs with high LD to that annotation have higher $\chi^2$ statistics than SNPs with low LD to it.

Let $a_{jC}$ denote the value of annotation $C$ at SNP $j$:

- **Binary annotation** (e.g. an indicator for "in enhancer", "in exon", "in cell-type-specific peak"): $a_{jC} \in \{0, 1\}$.
- **Continuous annotation** (e.g. gene-specificity score, conservation score, continuous epigenomic signal): $a_{jC} \in \mathbb{R}$.

Under a polygenic model the per-SNP heritability for SNP $j$ is

$$\mathrm{Var}(\beta_j) \;=\; \sum_C a_{jC}\, \tau_C$$

and the expected $\chi^2$ statistic of SNP $j$ is

$$E[\chi^2_j \mid \mathbf{a}_j] \;=\; N \sum_C \tau_C\, \ell(j, C) \;+\; N a \;+\; 1 \quad (2)$$

where $\ell(j, C) = \sum_k a_{kC}\, r^2_{jk}$ is the partitioned LD score of SNP $j$ with respect to annotation $C$, and $a$ measures confounding bias. Equation (2) allows joint estimation of all $\tau_C$ via a (computationally simple) multiple regression of $\chi^2_j$ against $\ell(j, C)$.

Interpretation of $\tau_C$:
- **Binary $C$**: $\tau_C$ is the *additive increase in per-SNP heritability* for SNPs in category $C$, on top of the contributions from any other annotations they belong to.
- **Continuous $C$**: $\tau_C$ is the *additive change in per-SNP heritability per unit increase* in the value of annotation $C$.

For application to real data and comparisons to other methods, see the three papers cited at the top of this notebook.

### Tau Estimation and Enrichment Analysis

Goal: quantify the contribution of functional annotations to trait heritability and assess statistical significance, accounting for LD structure and (for continuous annotations) annotation scale.

The pipeline has two computational layers:

- **Regression layer** — the S-LDSC regression itself, performed by the [polyfun](https://github.com/omerwe/polyfun) engine. We do not re-implement this.
- **Post-processing layer** — standardization, differential per-SNP heritability with proper jackknife inference, binary/continuous detection, and random-effects meta-analysis across traits. Implemented by us in the [`pecotmr`](https://github.com/StatFunGen/pecotmr) R package (`R/sldsc_wrapper.R`).

The notation below tags each modeling quantity as **(polyfun)** or **(pecotmr)** so the boundary is unambiguous.

#### Notation

For each annotation $C$ we use:

- $\pi^{h^2}_C$ = proportion of trait heritability $h^2_g$ assigned to annotation $C$.
- $\pi^{M}_C$ = proportion of (effective) SNPs in annotation $C$. For binary annotations this is $M_C / M_{\mathrm{ref}}$; for continuous annotations it is the share of total annotation weight in $C$.

#### Reference panel and MAF cutoff

All LD-derived quantities — partitioned LD scores for the 97 baseline annotations and for our $K$ target annotations, the LD-score-regression weights, allele frequencies, and the SNP set — are computed against our own LD reference panel. We do not mix in pre-computed quantities from external panels (e.g. 1000G); $M_{\mathrm{ref}}$ throughout this notebook denotes the number of common SNPs in our panel.

By default we restrict to MAF $> 5\%$ per the sLDSC recommendation: rare-variant LD is unstable and HapMap3-style regression weights are common-variant by construction. The cutoff is exposed as the SoS parameter `maf_cutoff` (default $0.05$); the regression, the standardized $sd_C$, and $M_{\mathrm{ref}}$ are all evaluated on the same MAF $>$ cutoff SNP set. If allele-frequency files are not available the pipeline fails; the user must explicitly set `maf_cutoff = 0` to opt out (not recommended).

#### Quantities from the regression layer (polyfun)

Solving Equation (2) jointly across annotations, with 200-block genomic jackknife for inference, is performed by polyfun's `ldsc.py`. From each polyfun run we obtain, per annotation:

- $\tau_C$ and its standard error — **(polyfun)**.
- $\pi^{h^2}_C$ and $\pi^{M}_C$ — **(polyfun)**.
- $E_C = \pi^{h^2}_C / \pi^{M}_C$ and its standard error — **(polyfun)**.
- The p-value of the differential per-SNP heritability test (defined below) — **(polyfun)**, computed internally with the full coefficient covariance matrix.

We also obtain, per run:

- The total trait heritability $h^2_g$ — **(polyfun)**.
- The 200-block jackknife delete-values of $\tau_C$ — **(polyfun)**.

#### Quantities from the post-processing layer (pecotmr)

From the polyfun outputs above plus our reference panel, the post-processing layer computes:

- $sd_C$ — per-annotation standard deviation over MAF $>$ cutoff SNPs — **(pecotmr: `compute_sldsc_annot_sd`)**.
- $M_{\mathrm{ref}}$ — reference SNP count at the MAF cutoff — **(pecotmr: `compute_sldsc_M_ref`)**.
- Whether each annotation is binary or continuous — **(pecotmr: `is_binary_sldsc_annot`)**.
- $\tau^*_C$ and per-block $\tau^*_C$ — **(pecotmr: `standardize_sldsc_trait`)**, formula given below.
- Per-block EnrichStat values and their jackknife standard error — **(pecotmr: `standardize_sldsc_trait`)** (only the point p-value is from polyfun).
- DerSimonian-Laird random-effects meta-analysis of $\tau^*_C$, $E_C$, or EnrichStat across traits — **(pecotmr: `meta_sldsc_random`)**.

The top-level entry point `pecotmr::sldsc_postprocessing_pipeline` orchestrates all of the above.

#### Standardized tau ($\tau^*$)  —  (pecotmr)

$\tau_C$ has units that depend on the scale of the annotation and on the total heritability of the trait, so raw $\tau$ is not directly comparable across annotations or across traits. We compute the standardized version (Gazal et al. 2017)

$$\tau^*_C \;=\; \tau_C \cdot \frac{sd_C \cdot M_{\mathrm{ref}}}{h^2_g}$$

interpreted as the additive change in per-SNP heritability associated with a 1 standard deviation increase in annotation $C$, divided by the average per-SNP heritability across all SNPs. $\tau^*_C$ is dimensionless and comparable across annotations and across traits. In a joint multi-annotation regression it is the *independent contribution* of annotation $C$ after controlling for overlapping effects of the others.

Here $sd_C$ is the standard deviation of annotation $C$ across reference SNPs (MAF $>$ cutoff), $M_{\mathrm{ref}}$ is the count of those SNPs, and $h^2_g$ is the trait heritability. Applying the same scaling to each of the 200 jackknife blocks yields per-block $\tau^*_C$ values, which provide the standard error needed for cross-trait meta-analysis.

#### Differential per-SNP heritability ("EnrichStat")  —  (polyfun + pecotmr)

To test whether the per-SNP heritability *inside* annotation $C$ differs from *outside* it (Finucane et al. 2015):

$$\text{EnrichStat}_C \;=\; \frac{h^2_g}{M_{\mathrm{ref}}} \!\left[\, \frac{\pi^{h^2}_C}{\pi^{M}_C} \;-\; \frac{1 - \pi^{h^2}_C}{1 - \pi^{M}_C} \,\right]$$

The point-estimate p-value of this test is computed by polyfun internally with the full coefficient covariance and reported as `Enrichment_p`. The post-processing layer (pecotmr) recomputes the per-block EnrichStat from the jackknife $\tau$ values to obtain a per-block standard error for use in cross-trait meta-analysis.

#### Reporting: binary vs. continuous annotations  —  (pecotmr)

The estimation machinery applies to both annotation types, but the *headline* quantity to report **within each type** differs.

For a **binary annotation** (e.g. enhancer indicator, exon, in/out of a cell-type peak), $\pi^{M}_C = M_C / M_{\mathrm{ref}}$ has a direct interpretation and $E_C$ reads as "the category explains $E_C$-fold more heritability than its share of SNPs." The within-type headline quantities are therefore $E_C$ and the EnrichStat p-value; $\tau^*_C$ is reported alongside.

For a **continuous annotation** (e.g. gene-specificity score, conservation score, continuous epigenomic signal), $E_C$ depends on the scale of the annotation: rescaling the annotation by a constant changes $E_C$ even though the underlying biology is unchanged. The within-type headline quantities are therefore $\tau^*_C$ and its p-value; $E_C$ is reported alongside but should not be compared across continuous annotations of different distributions.

The pipeline determines whether an annotation is binary by inspecting whether its values lie in $\{0, 1\}$ and selects the appropriate within-type headline statistic automatically (pecotmr).

#### Cross-type comparison: always use $\tau^*_C$  —  (pecotmr)

For an apple-to-apple comparison **across binary and continuous annotations** — ranking annotations on a single axis, meta-analyzing a mixed set, or reporting a leaderboard that pools both types — use $\tau^*_C$. The standardization in Gazal et al. (2017) was designed for exactly this purpose: $sd_C = \sqrt{p(1-p)}$ for a binary annotation (where $p$ is the proportion in the category) and $sd_C = $ empirical standard deviation for a continuous annotation, so the resulting $\tau^*_C$ is dimensionless and has the same interpretation in both cases — additive change in per-SNP heritability per 1 SD increase in the annotation, normalized by the average per-SNP heritability. $E_C$ does not have this property and should not be compared across types.

The pipeline emits both $E_C$ and $\tau^*_C$ for every annotation, with the binary/continuous flag, so callers can pick the right column for the comparison they are making.

#### Joint analysis  —  (polyfun runs the regression; pecotmr standardizes both modes)

For **joint analysis** (multiple annotations fit together), both $\tau$ and $E$ are conditional on the other annotations in the model. We report joint $\tau^*_C$ as the independent contribution of annotation $C$ after controlling for the others. Single-annotation and joint analyses are produced in one pipeline pass: for $N$ target annotations, the single-annotation results come from $N$ polyfun regressions of one target + baseline each, and the joint results come from one polyfun regression of all $N$ targets + baseline together. The post-processing layer reads all $N+1$ regression outputs and presents single + joint side-by-side.

### Meta-Analysis across Traits (Random Effects)  —  (pecotmr)

DerSimonian-Laird random-effects meta-analysis of per-annotation estimates across traits, implemented in `pecotmr::meta_sldsc_random` (which delegates the numerics to `rmeta::meta.summaries(..., method = "random")`):

$$\hat\theta_{\mathrm{meta}} \;=\; \frac{\sum_i w_i\, \hat\theta_i}{\sum_i w_i}, \qquad SE_{\mathrm{meta}} \;=\; \sqrt{\frac{1}{\sum_i w_i}}, \qquad w_i \;=\; \frac{1}{SE_i^2 + \hat\sigma^2}$$

where $\hat\theta_i$ is the per-trait estimate ($\tau^*_C$ or $E_C$ or EnrichStat), $SE_i$ its (jackknife or polyfun-reported) standard error, and $\hat\sigma^2$ the DerSimonian-Laird estimate of between-trait variance.

For enrichment meta-analysis the meta-analyzed $E_C$ provides the effect size and SE; the meta-analyzed EnrichStat provides the p-value. The pipeline produces a default meta over all supplied traits; users can re-run meta on any subset of traits without re-running the regression layer (see workflow Step 3).

$$Z_{\mathrm{meta}} \;=\; \frac{\hat\theta_{\mathrm{meta}}}{SE_{\mathrm{meta}}}, \qquad p \;=\; 2\,\Phi(-|Z_{\mathrm{meta}}|)$$

## Input

### 1. Target Annotation File

- **Purpose**: Specifies the user-provided ("target") genome annotation files. The pipeline supports both binary and continuous annotations; the type is auto-detected per annotation column.
- **Formats**:
    - Text file (`.txt`) listing per-chromosome paths to annotation files. Annotation files can be `.rds`/`.tsv`/`.txt`.
    - Alternatively, files for specific chromosomes can be provided directly.
    - **Multiple target annotations** are supported in one input file (one column per annotation, prefixed `path`, `path1`, `path2`, ...). Single-target and joint-target analyses are produced automatically in one pipeline pass.
    - **Format** (the score column is optional; if absent, score is set to 1):
        - `is_range = False`:
        ```
        chr   pos   score
        1    10001   1
        1    10002   1
        ```
        - `is_range = True`:
        ```
        chr   start   end   score
        1    10001  20001  1
        1    30001  40001  1
        ```

### 2. Reference Annotation File (baseline-LD)

- **Purpose**: Provides the baseline annotations (typically the 97-annotation baseline-LD model from Gazal et al. 2017) in `.annot.gz` format for each chromosome. The baseline conditions every regression.
- **Formats**:
    - Text file listing baseline annotation files for all chromosomes.
    - Alternatively, files for specific chromosomes can be provided directly.

### 3. Genome Reference File

- **Purpose**: PLINK-format `.bed/.bim/.fam` files for our LD reference panel, per chromosome. This is the panel against which all LD-derived quantities (target LD scores, baseline LD scores, regression weights, allele frequencies) must be computed. **Do not mix files derived from different panels** (e.g. 1000G vs ADSP).
- **Formats**:
    - Text file listing per-chromosome reference files, or files for specific chromosomes.

### 4. SNP List

- **Purpose**: Specifies the SNPs to include in LDSC analysis (typically a HapMap3-style list).
- **Format**: A list of `rsid`s, one per line.

### 5. Allele Frequency Files (`.frq`, our panel)

- **Purpose**: PLINK `.frq` files for the reference panel, used to enforce the MAF cutoff. **Required** when `maf_cutoff > 0` (default `0.05`); the pipeline fails if missing unless `maf_cutoff = 0` is explicitly set.

### 6. GWAS Summary Statistics

- **Purpose**: One munged sumstats file per trait, listed in a text file (`all_traits_file`). The pipeline runs the regression once per trait per single/joint mode.
- **Format**:
    ```
    CAD_META.filtered.sumstats.gz
    UKB.Lym.BOLT.sumstats.gz
    ```

---

## Workflow Steps

The pipeline is organized in three logical stages: (1) input preparation and the S-LDSC regression itself (handled by polyfun), (2) post-processing into standardized $\tau^*$ and meta-analysis (handled by R wrappers in the `pecotmr` package), and (3) optional re-meta on user-defined trait subsets.

### Step 1: Annotation Preparation and S-LDSC Regression (polyfun)

Two SoS workflows wrap polyfun:

- `make_annotation_files_ldscore` — for each target annotation, computes `.annot.gz` and `.l2.ldscore.parquet`/`.l2.M`/`.l2.M_5_50` against our reference panel. Computed once per target annotation; cached on disk and reused across all traits and all polyfun runs via SoS's output-timestamp check.
- `munge_sumstats_polyfun` — preprocesses each GWAS into LDSC-compatible format.
- `get_heritability` — for each trait, runs polyfun's `ldsc.py` once per single-target analysis ($N$ runs, each fitting 1 target + baseline) and once for the joint analysis (1 run, fitting all $N$ targets + baseline together). MAF $>$ cutoff is enforced via `--frqfile-chr`. Produces `.results`, `.log`, `.part_delete` per polyfun invocation.

### Step 2: Post-Processing (`pecotmr::sldsc_postprocessing_pipeline`)

A single R function call consumes all polyfun outputs for the run and produces the final tables:

- Reads each polyfun output and extracts $\tau$, $E$, $h^2_g$, EnrichStat p-value, and per-block jackknife $\tau$ values.
- Computes annotation $sd_C$ and $M_{\mathrm{ref}}$ over the same MAF $>$ cutoff SNP set as the regression.
- Standardizes $\tau \to \tau^*$ for both single-tau and joint-tau modes, including the per-block versions for jackknife SE.
- Auto-detects whether each annotation is binary or continuous and tags every output row accordingly.
- Reports the number and names of baseline annotations encountered (via `message()`) for transparency.
- Runs the default DerSimonian-Laird random-effects meta-analysis across all supplied traits, producing three meta tables: $\tau^*$ (cross-type comparable), $E$ (within-binary), and EnrichStat (within-type).

Outputs are returned as an R list with two top-level entries: `per_trait` (one tidy data frame per trait, single + joint estimates side-by-side per target) and `meta` (three tables, one per quantity, with rows = target annotations and columns = single/joint mean/SE/p plus an `is_binary` flag).

### Step 3: Optional Subset Meta-Analysis (`pecotmr::meta_sldsc_random`)

The default meta in Step 2 pools all traits the user supplied. To re-run the meta on a subset (e.g., neurodegenerative traits only, or autoimmune traits only) without re-running the regression layer:

```r
res <- readRDS("sldsc_results.rds")
neuro <- c("AD_GWAX", "PD_meta", "ALS_meta")
meta_neuro_taustar <- pecotmr::meta_sldsc_random(
  res$per_trait[neuro], category = "my_target_anno", quantity = "tau_star"
)
```

This step is light-weight and can be run interactively.

---

### Output summary

| Stage | Cached on disk | Recomputable from | Purpose |
|---|---|---|---|
| Target LD scores | per-annotation, once | annotation + reference panel | input to every regression |
| polyfun `.results` per (trait, mode) | yes | regression run | $\tau$, $E$, EnrichStat |
| Per-trait standardized table | yes (RDS) | polyfun outputs + $sd_C$ + $M_{\mathrm{ref}}$ | reporting + meta |
| Default meta tables | yes (RDS) | per-trait standardized | headline figures |
| Subset meta | re-run on demand | per-trait standardized | custom analyses |

## MWE: 
### 1. make_annotation_files_ldscore
annotation file can be a txt file with #id, and path1 path2 ..., also can be rds files seperate by ',' 
#### 1.1 single tau analysis, with one annotation as a input

In [ ]:
 # case 1: txt file as input
 sos run pipeline/sldsc_enrichment.ipynb make_annotation_files_ldscore \
    --annotation_file data/polyfun/input/colocboost_test_annotation_path.txt \
    --reference_anno_file data/polyfun/input/reference_annotation0.txt \
    --genome_ref_file data/polyfun/input/genome_reference_bfile.txt \
    --annotation_name test_colocboost \
    --plink_name reference. \
    --baseline_name annotations. \
    --weight_name weights. \
    --python_exec python \
    --polyfun_path data/github/polyfun \
    --cwd output/polyfun/ -j 22

Alternatively, we can also use files with specific chromosome, instead of txt list.

In [ ]:
# single file format
 sos run pipeline/sldsc_enrichment.ipynb make_annotation_files_ldscore \
    --annotation_file data/polyfun/input/colocboost_test.tsv \
    --reference_anno_file data/polyfun/example_annot0/annotations.1.annot.gz \
    --genome_ref_file data/polyfun/example_data/reference.1.bed \
    --annotation_name test_colocboost \
    --plink_name reference. \
    --baseline_name annotations. \
    --weight_name weights. \
    --python_exec python \
    --polyfun_path data/github/polyfun \
    --cwd output/polyfun/ --chromosome 1

##### 1.2 joint tau
with more than one annotation as the input

In [ ]:
# input case1: txt file format
 sos run /home/al4225/project/quantile_twas/analysis/SLDSC/scripts/ldsc.ipynb make_annotation_files_ldscore \
    --annotation_file /home/al4225/project/quantile_twas/analysis/SLDSC/data/quantile_qtl_annotation/test/AC_DeJager_eQTL.joint_tau_example.txt \
    --reference_anno_file /home/al4225/project/quantile_twas/analysis/SLDSC/data/reference_annotation.txt \
    --genome_ref_file /home/al4225/project/quantile_twas/analysis/SLDSC/data/genome_reference_bfile.txt \
    --snp_list /mnt/vast/hpc/csg/xc2270/colocboost/post/SLDSC/1000G_EUR_Phase3_hg38/list.txt \
    --annotation_name joint_test \
    --cwd /home/al4225/project/quantile_twas/analysis/SLDSC/output --joint_tau --chromosome 1

In [ ]:
 # input case 2: single file format, annotation files separate by ','.
 sos run /home/al4225/project/quantile_twas/analysis/SLDSC/scripts/ldsc.ipynb make_annotation_files_ldscore \
    --annotation_file /home/al4225/project/quantile_twas/analysis/output/157genes/step2_rq_lr_summary/enrich_info_by_context_class/AC_DeJager_eQTL.unique_qr.rds,/home/al4225/project/quantile_twas/analysis/output/157genes/step2_rq_lr_summary/enrich_info_by_context_class/AC_DeJager_eQTL.shared_homo.rds \
    --reference_anno_file /mnt/vast/hpc/csg/xc2270/colocboost/post/SLDSC/example_anno/ABC_Road_GI_BRN.1.annot.gz \
    --genome_ref_file /mnt/vast/hpc/csg/xc2270/colocboost/post/SLDSC/plink_files/1000G.EUR.hg38.1.bed \
    --snp_list /mnt/vast/hpc/csg/xc2270/colocboost/post/SLDSC/1000G_EUR_Phase3_hg38/list.txt \
    --annotation_name joint_test \
    --cwd /home/al4225/project/quantile_twas/analysis/SLDSC/output/mwe/joint_tau --joint_tau --chromosome 1

#### 2. get_heritability

In [ ]:
sos run pipeline/sldsc_enrichment.ipynb get_heritability \
    --target_anno_dir output/polyfun/test_colocboost \
    --sumstat_dir data/polyfun/example_data \
    --baseline_ld_dir data/polyfun/example_data \
    --python_exec python \
    --polyfun_path data/github/polyfun \
    --weights_dir data/polyfun/example_data \
    --plink_name reference. \
    --baseline_name annotations. \
    --weight_name weights. \
    --annotation_name test_colocboost \
    --cwd output/polyfun/test_colocboost/sumstats/ \
    --all_traits_file data/polyfun/input/sumstats_test_all.txt \
    -s build -j 2

### 3. processed_stats
#### 3.1 single tau analysis, with one annotation as a input

In [ ]:
# processed stats cwd has to be the same with get_heritability
sos run pipeline/sldsc_enrichment.ipynb processed_stats \
    --target_anno_dir output/polyfun/test_colocboost \
    --sumstat_dir data/polyfun/example_data \
    --baseline_ld_dir data/polyfun/example_data \
    --python_exec python \
    --polyfun_path data/github/polyfun \
    --weights_dir data/polyfun/example_data \
    --plink_name reference. \
    --baseline_name annotations. \
    --weight_name weights. \
    --annotation_name test_colocboost \
    --cwd output/polyfun/test_colocboost/sumstats/ \
    --trait_group_paths "data/polyfun/input/sumstats_test_all.txt data/polyfun/input/sumstats_test_category1.txt" \
    --trait_group_names "All category1" \
    --all_traits_file data/polyfun/input/sumstats_test_all.txt


##### 3.2 joint tau
with more than one annotation as the input

In [ ]:
# processed stats cwd has to be the same with get_heritability
sos run /home/al4225/project/quantile_twas/analysis/SLDSC/scripts/ldsc.ipynb processed_stats \
    --target_anno_dir /home/al4225/project/quantile_twas/analysis/SLDSC/output/joint_test/ \
    --baseline_ld_dir /mnt/vast/hpc/csg/xc2270/colocboost/post/SLDSC/1000G_EUR_Phase3_hg38/baselineLD_v2.2 \
    --annotation_name joint_test \
    --cwd /home/al4225/project/quantile_twas/analysis/SLDSC/output/joint_test/heritability \
    --trait_group_paths "/home/al4225/project/quantile_twas/analysis/SLDSC/data/sumstats_test_all.txt /home/al4225/project/quantile_twas/analysis/SLDSC/data/sumstats_test_brain_trait.txt /home/al4225/project/quantile_twas/analysis/SLDSC/data/sumstats_test_blood_trait.txt" \
    --trait_group_names "All Brain Blood" \
    --all_traits_file /home/al4225/project/quantile_twas/analysis/SLDSC/data/sumstats_test_all.txt -s build --joint_tau --joint_annotation_names "AC.eqtl.unique_qr AC.eqtl.shared_heter"

In [ ]:
[global]
# Path to the work directory of the analysis.
parameter: cwd = path('output')
# Prefix for the analysis output
parameter: annotation_name = str
parameter: joint_annotation_names = [] #if joint analysis is used, pass annotation names
parameter: joint_tau = False
parameter: python_exec = "python" # e.g. "/home/you/.conda/envs/polyfun/bin/python"
parameter: polyfun_path   = path # e.g. "/home/you/tools/polyfun"

# MAF cutoff for sLDSC. Default 0.05 per sLDSC recommendation (rare-variant LD is unstable
# and HapMap3-style regression weights are common-variant by construction).
# Set to 0 to opt out of MAF filtering (NOT recommended; only use if you understand the implications).
# Other values would require recomputing LD scores at that cutoff.
parameter: maf_cutoff = 0.05

# for make_annotation_files_ldscore workflow:
parameter: annotation_file = path()
parameter: reference_anno_file = path()
parameter: genome_ref_file = path() # with .bed
parameter: chromosome = []
parameter: snp_list = path()
parameter: ld_wind_kb = 0 # use kb if the value is provided
parameter: ld_wind_cm = 1.0 # default using ld_wind_cm

# for get_heritability workflow.
# Note: all LD-derived inputs (baseline LD scores, target LD scores, regression weights,
# allele frequencies) must be computed against the same reference panel as `genome_ref_file`.
# Do not mix files derived from different reference panels (e.g., 1000G vs ADSP).
parameter: all_traits_file = path() # txt file, each row contains all GWAS summary statistics name: e.g. CAD_META.filtered.sumstats.gz
parameter: sumstat_dir = path() # Directory containing GWAS summary statistics
parameter: target_anno_dir = path()  # Directory containing target annotation files: output of ldscore
parameter: baseline_ld_dir = path()  # Directory containing baseline LD score files (computed against our panel)
parameter: frqfile_dir = path()  # Directory containing allele frequency files (.frq, our panel)
parameter: plink_name = "ADSP_chr"
parameter: weights_dir = path()  # Directory containing LD weights (computed against our panel)
parameter: baseline_name = "baseline_chr"  # Prefix of baseline annotation files
parameter: weight_name = "weights_chr"  # Prefix of LD weights files
parameter: Mref = -1 # Reference number of SNPs in our panel (MAF > maf_cutoff). If > 0, use provided value; if <= 0, auto-calculate from ldscore files
parameter: n_blocks = 200

# Number of threads
parameter: numThreads = 16
# For cluster jobs, number commands to run per job
parameter: job_size = 1
parameter: walltime = '12h'
parameter: mem = '16G'

## Make Annotation File

In [ ]:
[make_annotation_files_ldscore]
parameter: score_column = 3
parameter: is_range = False # parameter to handle range expansion

import pandas as pd
import os

def adapt_file_path(file_path, reference_file):
    """
    Adapt a single file path based on its existence and a reference file's path.

    Args:
    - file_path (str): The file path to adapt.
    - reference_file (str): File path to use as a reference for adaptation.

    Returns:
    - str: Adapted file path.

    Raises:
    - FileNotFoundError: If no valid file path is found.
    """
    reference_path = os.path.dirname(reference_file)

    # Check if the file exists
    if os.path.isfile(file_path):
        return file_path

    # Check file name without path
    file_name = os.path.basename(file_path)
    if os.path.isfile(file_name):
        return file_name

    # Check file name in reference file's directory
    file_in_ref_dir = os.path.join(reference_path, file_name)
    if os.path.isfile(file_in_ref_dir):
        return file_in_ref_dir

    # Check original file path prefixed with reference file's directory
    file_prefixed = os.path.join(reference_path, file_path)
    if os.path.isfile(file_prefixed):
        return file_prefixed

    # If all checks fail, raise an error
    raise FileNotFoundError(f"No valid path found for file: {file_path}")


def adapt_file_path_all(df, column_name, reference_file):
    return df[column_name].apply(lambda x: adapt_file_path(x, reference_file))

# Process input files based on file type
if (str(annotation_file).endswith(('rds', 'tsv', 'txt', 'tsv.gz', 'txt.gz')) and 
    str(reference_anno_file).endswith('annot.gz')):
    # Case 1: Direct file paths
    # joint tau: annotation_fills: rds/tsv/txt files separete with ","    
    if joint_tau:
        target_files = str(annotation_file).split(',')
        input_files = [[*target_files, reference_anno_file, genome_ref_file]]
    else:
        # single annotation
        input_files = [[annotation_file, reference_anno_file, genome_ref_file]]
    
    if len(chromosome) > 0:
        input_chroms = [int(x) for x in chromosome]
    else:
        input_chroms = [0]
else:
    # Case 2: Files with #id and cols starting with path columns
    target_files = pd.read_csv(annotation_file, sep="\t")
    reference_files = pd.read_csv(reference_anno_file, sep="\t")
    genome_ref_files = pd.read_csv(genome_ref_file, sep="\t")
    
    # Standardize #id 
    target_files["#id"] = [x.replace("chr", "") for x in target_files["#id"].astype(str)]
    reference_files["#id"] = [x.replace("chr", "") for x in reference_files["#id"].astype(str)]
    genome_ref_files["#id"] = [x.replace("chr", "") for x in genome_ref_files["#id"].astype(str)]
    
    # process target annotation files
    path_columns = [col for col in target_files.columns if col.startswith('path')]
    for col in path_columns:
        target_files[col] = target_files[col].apply(lambda x: adapt_file_path(x, annotation_file))
    
    # process reference and genome files
    reference_files["path"] = reference_files["path"].apply(lambda x: adapt_file_path(x, reference_anno_file))
    genome_ref_files["path"] = genome_ref_files["path"].apply(lambda x: adapt_file_path(x, genome_ref_file))
    
    # Merge the files based on #id
    input_files = target_files.merge(reference_files, on="#id").merge(genome_ref_files, on="#id")
    
    # Filter by specified chromosomes, if any
    if len(chromosome) > 0:
        input_files = input_files[input_files['#id'].isin(chromosome)]
    
    # Extract relevant columns as a list of file paths
    input_files = input_files.values.tolist()
    input_chroms = [x[0] for x in input_files]  # Chromosome IDs
    
    if joint_tau:
        # joint tau, keep all path columns
        input_files = [[*x[1:len(path_columns)+1], x[-2], x[-1]] for x in input_files]
    else:
        # single annotation
        input_files = [[x[1], x[-2], x[-1]] for x in input_files]
        
input: input_files, group_by = len(input_files[0]), group_with = "input_chroms"
# Dynamically determine output format based on snp_list parameter
import os
use_print_snps = snp_list.is_file()

if use_print_snps:
    # ldsc.py outputs .gz format
    ldscore_ext = "l2.ldscore.gz"
else:
    # compute_ldscores.py outputs .parquet format  
    ldscore_ext = "l2.ldscore.parquet"

if ld_wind_kb > 0:
    use_kb_window = True
    ld_window_param = ld_wind_kb
    ld_window_flag = "--ld-wind-kb"
else:
    use_kb_window = False
    ld_window_param = ld_wind_cm 
    ld_window_flag = "--ld-wind-cm"

output: dict([
   ('annot', f'{cwd:a}/{annotation_name}/{annotation_name}.{input_chroms[_index]}.annot.gz'),
   ('ldscore', f'{cwd:a}/{annotation_name}/{annotation_name}.{input_chroms[_index]}.{ldscore_ext}'),
   ('Mfile',  f'{cwd:a}/{annotation_name}/{annotation_name}.{input_chroms[_index]}.l2.M') # set output for generation .M files
])

task: trunk_workers = 1, trunk_size = job_size, walltime = walltime, mem = mem, cores = numThreads, tags = f'{step_name}_{_output[0]:bnn}'

R: expand= "${ }", stderr = f'{_output["annot"]}.stderr', stdout = f'{_output["annot"]}.stdout'
    library(data.table)

    # Function to clean chromosome notation
    clean_chr <- function(x) {
        as.numeric(gsub("^chr", "", x))
    }

    # Function to process range data - only expands ranges for the specified chromosome
    process_range_data <- function(data, chr_value) {
        # First filter for the specific chromosome
        data$chr <- clean_chr(data$chr)
        data <- data[data$chr == chr_value,]
        
        if(nrow(data) == 0) {
            return(NULL)
        }
        
        # Only expand ranges for the matched chromosome
        expanded_rows <- lapply(1:nrow(data), function(j) {
            row <- data[j,]
            pos_seq <- seq(row$start, row$end-1)
            
            # Create base data frame with chr and pos
            result <- data.frame(
                chr = rep(row$chr, length(pos_seq)),
                pos = pos_seq
            )
            
            # Add additional columns if they exist
            if(ncol(data) > 3) {
                for(col in 4:ncol(data)) {
                    result[[names(data)[col]]] <- rep(row[[col]], length(pos_seq))
                }
            }
            
            return(result)
        })
        result <- unique(rbindlist(expanded_rows))
        return(result)
    }

    # Main annotation processing function
    process_annotation <- function(target_anno, ref_anno, score_column_value) {
        target_anno <- as.data.frame(target_anno)
        ref_anno <- as.data.frame(ref_anno)
        
        # Clean chromosome notation
        target_anno$chr <- clean_chr(target_anno$chr)
        ref_anno$CHR <- clean_chr(ref_anno$CHR)
        
        chr_value = unique(ref_anno$CHR)
        pos <- which(target_anno$chr == chr_value)
        anno_scores <- rep(0, nrow(ref_anno))
        
        match_pos <- match(target_anno$pos, ref_anno$BP)
        valid_pos <- as.numeric(na.omit(match_pos))
        
        if (score_column_value <= ncol(target_anno)) {
            anno_scores[valid_pos] <- target_anno[[score_column_value]][!is.na(match_pos)]
        } else {
            anno_scores[valid_pos] <- 1
            print("Warning: Score column does not exist. Setting scores to 1")
        }
        
        return(anno_scores)
    }

    # Read reference annotation file
    ref_anno <- fread(${_input[-2]:ar})
    ref_anno <- as.data.frame(ref_anno)
    if("ANNOT" %in% colnames(ref_anno)) {
        ref_anno <- ref_anno[,-which(colnames(ref_anno) == "ANNOT")]
    }
    result_anno <- ref_anno

    score_column_value <- ${score_column}  
    is_joint <- ${"TRUE" if joint_tau else "FALSE"}
    is_range <- ${"TRUE" if is_range else "FALSE"}

    if(is_joint) {
        target_files = c(${",".join(['"%s"' % x.absolute() for x in _input[:-2]])})
        for(i in seq_along(target_files)) {
            # Read file based on format
            if(endsWith(target_files[i], "rds")) {
                target_anno <- readRDS(target_files[i])
            } else {  # Handle TSV/TXT files
                target_anno <- fread(target_files[i])
                
                # Process range data if needed
                if(is_range) {
                    names(target_anno)[1:3] <- c("chr", "start", "end")
                    target_anno <- process_range_data(target_anno, unique(ref_anno$CHR))
                    if(is.null(target_anno)) {
                        anno_scores <- rep(0, nrow(ref_anno))
                    } else {
                        anno_scores <- process_annotation(target_anno, ref_anno, score_column_value)
                    }
                } else {
                    names(target_anno)[1:2] <- c("chr", "pos")
                    anno_scores <- process_annotation(target_anno, ref_anno, score_column_value)
                }
            }
            
            print(paste("Processing annotation file", i))
            result_anno[[paste0("ANNOT_", i)]] <- anno_scores
        }
    } else {
        # Read file based on format
        if(endsWith(${_input[0]:ar}, "rds")) {
            target_anno <- readRDS(${_input[0]:ar})
        } else {  # Handle TSV/TXT files
            target_anno <- fread(${_input[0]:ar})
            
            # Process range data if needed
            if(is_range) {
                names(target_anno)[1:3] <- c("chr", "start", "end")
                target_anno <- process_range_data(target_anno, unique(ref_anno$CHR))
                if(is.null(target_anno)) {
                    anno_scores <- rep(0, nrow(ref_anno))
                } else {
                    anno_scores <- process_annotation(target_anno, ref_anno, score_column_value)
                }
            } else {
                names(target_anno)[1:2] <- c("chr", "pos")
                anno_scores <- process_annotation(target_anno, ref_anno, score_column_value)
            }
        }
        
        result_anno$ANNOT <- anno_scores
    }

    # Write results
    fwrite(result_anno, ${_output["annot"]:nr}, 
        quote=FALSE, col.names=TRUE, row.names=FALSE, sep="\t")

bash: expand= "$[ ]", stderr = f'{_output["annot"]:nnn}.stderr', stdout = f'{_output["annot"]:nnn}.stdout'
   gzip -f $[_output["annot"]:n] 

bash: expand="${ }", stderr = f'{_output[1]}.stderr', stdout = f'{_output[1]}.stdout'
    echo "Running chromosome ${input_chroms[_index]}"
    echo "use_print_snps = ${use_print_snps}"
    echo "use_kb_window = ${use_kb_window}" 
    echo "ld_window_flag = ${ld_window_flag}"
    echo "ld_window_param = ${ld_window_param}"
    if [ "${use_kb_window}" = "True" ]; then
        echo "Using custom LD window: ${ld_window_param} kb"
    else
        echo "Using default LD window: ${ld_window_param} cM"
    fi
    
    # Use the dynamically determined mode
    if [ "${use_print_snps}" = "True" ]; then
        echo "SNP list found: ${snp_list}"
        echo "Using ldsc.py --print-snps mode"
        ${python_exec} ${polyfun_path}/ldsc.py \
          --print-snps ${snp_list} \
          ${ld_window_flag} ${ld_window_param} \
          --out ${_output["ldscore"]:nnn} \
          --bfile ${_input[-1]:nar} \
          --yes-really \
          --annot ${_output["annot"]:a} \
          --l2
    else
        echo "No SNP list provided, using compute_ldscores.py to save time"
        ${python_exec} ${polyfun_path}/compute_ldscores.py \
          --annot ${_output["annot"]:a} \
          --bfile ${_input[-1]:nar} \
          ${ld_window_flag} ${ld_window_param} \
          --out ${_output["ldscore"]} \
          --allow-missing
    fi

R: expand="${ }", stderr=f'{_output["Mfile"]}.stderr', stdout=f'{_output["Mfile"]}.stdout'
    # Check if --print-snps mode was used
    use_print_snps <- ${str(use_print_snps).upper()}
    out_file     <- ${_output["Mfile"]:ar}
    out_file_550 <- paste0(${_output["Mfile"]:ar}, "_5_50")
    
    if(use_print_snps) {
        cat("--print-snps mode was used, checking if M files already exist...\n")
        
        if(file.exists(out_file)) {
            cat("M file already exists:", out_file, "\n")
            if(file.exists(out_file_550)) {
                cat("M_5_50 file already exists:", out_file_550, "\n")
                cat("Both M files found, skipping generation\n")
                quit(save = "no", status = 0)
            }
        }
    }
    
    cat("Proceeding with M file generation...\n")

    # Generate .M and .M_5_50 files based on SNPs actually used in LD score computation
    suppressPackageStartupMessages({
        library(data.table)
        library(dplyr)
    })
    
    # File paths
    ldscore_file <- ${_output["ldscore"]:ar}   # polyfun generated .l2.ldscore.parquet
    annot_file   <- ${_output["annot"]:ar}     # annotation file
    bfile_prefix <- ${_input[-1]:nar}          # plink file prefix
    
    cat("Reading input files...\n")
    cat("LD score file path:", ldscore_file, "\n")
    
    # 1. Read LD scores file (polyfun output) - Fixed logic
    ldscore_dt <- NULL
    
    # Try reading the file based on its actual extension
    if(endsWith(ldscore_file, ".parquet")) {
        tryCatch({
            suppressPackageStartupMessages(library(arrow))
            ldscore_dt <- arrow::read_parquet(ldscore_file)
            cat("Successfully read LD scores from .parquet file\n")
        }, error = function(e) {
            cat("Error reading parquet file:", e$message, "\n")
            cat("Trying alternative methods...\n")
        })
    } else if(endsWith(ldscore_file, ".gz")) {
        tryCatch({
            ldscore_dt <- fread(ldscore_file)
            cat("Successfully read LD scores from .gz file\n")
        }, error = function(e) {
            cat("Error reading gz file:", e$message, "\n")
            cat("Trying alternative methods...\n")
        })
    } else {
        # Check if alternative file formats exist
        alt_files <- c(
            paste0(ldscore_file, ".parquet"),
            paste0(ldscore_file, ".gz"),
            paste0(gsub("\\.[^.]*$", "", ldscore_file), ".parquet"),
            paste0(gsub("\\.[^.]*$", "", ldscore_file), ".gz")
        )
        
        for(alt_file in alt_files) {
            if(file.exists(alt_file)) {
                cat("Found alternative file:", alt_file, "\n")
                tryCatch({
                    if(endsWith(alt_file, ".parquet")) {
                        suppressPackageStartupMessages(library(arrow))
                        ldscore_dt <- arrow::read_parquet(alt_file)
                        cat("Successfully read from alternative parquet file\n")
                        break
                    } else if(endsWith(alt_file, ".gz")) {
                        ldscore_dt <- fread(alt_file)
                        cat("Successfully read from alternative gz file\n")
                        break
                    }
                }, error = function(e) {
                    cat("Failed to read", alt_file, ":", e$message, "\n")
                })
            }
        }
        
        # Last resort: try direct read
        if(is.null(ldscore_dt) && file.exists(ldscore_file)) {
            tryCatch({
                ldscore_dt <- fread(ldscore_file)
                cat("Successfully read LD scores from direct file read\n")
            }, error = function(e) {
                cat("Error in direct file read:", e$message, "\n")
            })
        }
    }
    
    # Check if we successfully read the file
    if(is.null(ldscore_dt)) {
        stop("Failed to read LD scores file from any format. Please check the file: ", ldscore_file)
    }
    
    cat("LD scores data dimensions:", nrow(ldscore_dt), "x", ncol(ldscore_dt), "\n")
    cat("LD scores columns:", paste(names(ldscore_dt), collapse=", "), "\n")
    
    # 2. Read annotation file
    annot_dt <- fread(annot_file)
    cat(sprintf("Read annotation file with %d SNPs\n", nrow(annot_dt)))
    
    # 3. Check for .frq file and read MAF information if available
    frq_file <- paste0(bfile_prefix, ".frq")
    has_frq <- file.exists(frq_file)
    
    if(has_frq) {
        cat("Found .frq file, will calculate both M and M_5_50\n")
        frq_dt <- fread(frq_file)
        frq_dt <- frq_dt[, .(SNP, MAF)]
        cat(sprintf("Read MAF information for %d SNPs\n", nrow(frq_dt)))
    } else {
        cat("No .frq file found, will only calculate M file\n")
        frq_dt <- NULL
    }
    
    # 4. Filter based on SNPs actually used in LD scores computation
    # LD scores file contains the final set of SNPs used
    used_snps <- ldscore_dt$SNP
    cat(sprintf("LD scores file contains %d SNPs\n", length(used_snps)))
    
    # 5. Filter annotation file to keep only SNPs in LD scores
    annot_filtered <- annot_dt[annot_dt$SNP %in% used_snps, ]
    cat(sprintf("Annotation file after filtering: %d SNPs\n", nrow(annot_filtered)))
    
    # 6. Merge with MAF information if available
    if(has_frq) {
        annot_with_maf <- merge(annot_filtered, frq_dt, by = "SNP", all.x = TRUE)
        cat(sprintf("After merging with MAF: %d SNPs\n", nrow(annot_with_maf)))
    } else {
        annot_with_maf <- annot_filtered
    }
    
    # 7. Identify annotation columns (exclude standard columns)
    standard_cols <- c("CHR", "SNP", "BP", "CM", "A1", "A2")
    if(has_frq) {
        standard_cols <- c(standard_cols, "MAF")
    }
    
    annot_cols <- setdiff(names(annot_with_maf), standard_cols)
    
    # Add default annotation column if none exist
    if (length(annot_cols) == 0L) {
        annot_with_maf[, ANNOT := 1L]
        annot_cols <- "ANNOT"
        cat("No annotation columns found, added default ANNOT column\n")
    }
    
    cat(sprintf("Found %d annotation columns: %s\n", 
                length(annot_cols), paste(annot_cols, collapse=", ")))
    
    # 8. Calculate M values (all SNPs)
    M <- annot_with_maf[, lapply(.SD, sum, na.rm=TRUE), .SDcols = annot_cols]
    
    cat(sprintf("Total SNPs used for M calculation: %d\n", nrow(annot_with_maf)))
    
    # 9. Calculate M_5_50 values (MAF > 0.05 SNPs) if MAF data available
    if(has_frq) {
        annot_common <- annot_with_maf[!is.na(MAF) & MAF > 0.05, ]
        M_5_50 <- annot_common[, lapply(.SD, sum, na.rm=TRUE), .SDcols = annot_cols]
        cat(sprintf("Common SNPs (MAF>0.05) used for M_5_50: %d\n", nrow(annot_common)))
    }
    
    # 10. Write .l2.M file (ensure single row format)
    M_values <- as.numeric(M)
    writeLines(paste(M_values, collapse=" "), out_file)
    
    cat(sprintf("[SUCCESS] Wrote %s with M values: %s\n", 
                out_file, paste(M_values, collapse=" ")))
    
    # 11. Write .l2.M_5_50 file if MAF data available
    if(has_frq) {
        M_5_50_values <- as.numeric(M_5_50)
        writeLines(paste(M_5_50_values, collapse=" "), out_file_550)
        
        cat(sprintf("[SUCCESS] Wrote %s with M_5_50 values: %s\n", 
                    out_file_550, paste(M_5_50_values, collapse=" ")))
    } else {
        cat("[INFO] Skipped M_5_50 file generation (no MAF data available)\n")
    }
    
    cat("M file generation completed successfully!\n")


## Munge Summary Statistics

In [ ]:
# The script munge_polyfun_sumstats.py automatically detects whether signed statistics 
# (Z, BETA/SE, etc.) are present and computes Z-scores if needed. 
[munge_sumstats_polyfun]
parameter: sumstats  = path
parameter: n       = 0
parameter: min_info = 0.6
parameter: min_maf  = 0.001
parameter: keep_hla = False
parameter: chi2_cut = 30
input: sumstats
output: f"{_input:n}.munged.parquet"
bash: expand=True, stderr=f'{_output:nn}.stderr', stdout=f'{_output:nn}.stdout'
    {python_exec} {polyfun_path}/munge_polyfun_sumstats.py \
        --sumstats {_input} \
        --out {_output} \
        {'--n {}'.format(n) if n>0 else ''} \
        {'--min-info {}'.format(min_info)} \
        {'--min-maf {}'.format(min_maf)} \
        {'--chi2-cutoff {}'.format(chi2_cut)} \
        {'--keep-hla' if keep_hla else ''} \
        --remove-strand-ambig

## Calculate Functional Enrichment using Annotations

In [ ]:
[get_heritability]
# First read the traits file and create input paths
parameter: all_traits = []

import pandas as pd
import os

# Read traits and construct full paths
with open(all_traits_file, 'r') as f:
    all_traits = [os.path.join(sumstat_dir, line.strip()) for line in f if line.strip()]

input: all_traits, group_by = 1
output: f"{cwd}/{os.path.basename(_input[0])}.log", f"{cwd}/{os.path.basename(_input[0])}.results"
task: trunk_workers = 1, trunk_size = job_size, walltime = walltime, mem = mem, cores = numThreads, tags = f'{step_name}_{_output:bn}'

bash: expand = "${ }"
    trait="${os.path.basename(_input[0])}"
    output_dir="${cwd}"
    mkdir -p $output_dir

    # MAF cutoff handling (default 0.05; sLDSC recommendation).
    # If maf_cutoff > 0 we require .frq files for the panel and use --frqfile-chr,
    # which makes polyfun read M_5_50 and operate on MAF > 5% SNPs only.
    # If maf_cutoff == 0 we use --not-M-5-50 (all SNPs); the user must opt in explicitly.
    frq_file_check="${frqfile_dir}/${plink_name}22.frq"
    if [ "${maf_cutoff}" = "0" ] || [ "${maf_cutoff}" = "0.0" ]; then
        echo "maf_cutoff = 0: skipping MAF filter (--not-M-5-50)"
        frq_option="--not-M-5-50"
    elif [ -f "$frq_file_check" ]; then
        echo "maf_cutoff = ${maf_cutoff}: using --frqfile-chr (MAF > 5%)"
        frq_option="--frqfile-chr ${frqfile_dir}/${plink_name}"
    else
        echo "ERROR: maf_cutoff=${maf_cutoff} requires .frq files for the reference panel,"
        echo "       but none found at ${frqfile_dir}/${plink_name}*.frq."
        echo "       Either provide .frq files for the panel, or explicitly set maf_cutoff=0 (NOT recommended)."
        exit 1
    fi

    # First attempt: original command
    ${python_exec} ${polyfun_path}/ldsc.py \
        --h2 ${sumstat_dir}/$trait \
        --ref-ld-chr ${target_anno_dir}/${annotation_name}.,${baseline_ld_dir}/${baseline_name} \
        --out ${cwd}/$trait \
        --overlap-annot \
        --w-ld-chr ${weights_dir}/${weight_name} \
        $frq_option \
        --print-coefficients \
        --print-delete-vals \
        --n-blocks ${n_blocks}

    # Check if it failed with FloatingPointError
    if [ $? -ne 0 ] && grep -q "FloatingPointError\|invalid value encountered in sqrt" "${cwd}/$trait.log" 2>/dev/null; then
        echo "FloatingPointError detected, retrying with --chisq-max 30..."

        ${python_exec} ${polyfun_path}/ldsc.py \
            --h2 ${sumstat_dir}/$trait \
            --ref-ld-chr ${target_anno_dir}/${annotation_name}.,${baseline_ld_dir}/${baseline_name} \
            --out ${cwd}/$trait \
            --overlap-annot \
            --w-ld-chr ${weights_dir}/${weight_name} \
            $frq_option \
            --chisq-max 30 \
            --print-coefficients \
            --print-delete-vals \
            --n-blocks ${n_blocks}

        # Check again if it still failed with FloatingPointError
        if [ $? -ne 0 ] && grep -q "FloatingPointError\|invalid value encountered in sqrt" "${cwd}/$trait.log" 2>/dev/null; then
            echo "Still FloatingPointError with --chisq-max 30, retrying with --chisq-max 20..."

            ${python_exec} ${polyfun_path}/ldsc.py \
                --h2 ${sumstat_dir}/$trait \
                --ref-ld-chr ${target_anno_dir}/${annotation_name}.,${baseline_ld_dir}/${baseline_name} \
                --out ${cwd}/$trait \
                --overlap-annot \
                --w-ld-chr ${weights_dir}/${weight_name} \
                $frq_option \
                --chisq-max 20 \
                --print-coefficients \
                --print-delete-vals \
                --n-blocks ${n_blocks}

            # Check a third time if it still failed with FloatingPointError
            if [ $? -ne 0 ] && grep -q "FloatingPointError\|invalid value encountered in sqrt" "${cwd}/$trait.log" 2>/dev/null; then
                echo "Still FloatingPointError with --chisq-max 20, retrying with --chisq-max 10..."

                ${python_exec} ${polyfun_path}/ldsc.py \
                    --h2 ${sumstat_dir}/$trait \
                    --ref-ld-chr ${target_anno_dir}/${annotation_name}.,${baseline_ld_dir}/${baseline_name} \
                    --out ${cwd}/$trait \
                    --overlap-annot \
                    --w-ld-chr ${weights_dir}/${weight_name} \
                    $frq_option \
                    --chisq-max 10 \
                    --print-coefficients \
                    --print-delete-vals \
                    --n-blocks ${n_blocks}

                # Final check - if still failing, log the persistent error
                if [ $? -ne 0 ] && grep -q "FloatingPointError\|invalid value encountered in sqrt" "${cwd}/$trait.log" 2>/dev/null; then
                    echo "ERROR: FloatingPointError persists even with --chisq-max 10 for trait: $trait"
                    echo "This trait may have severe numerical instability issues in the summary statistics."
                fi
            fi
        fi
    fi

In [ ]:
[processed_stats_1]
parameter: num_non_annotation_cols = -1 # number of non-annotation columns like CHR, BP, SNP, A1, A2, CM, MAF, default is calculate from pipeline, if provided, directly use this number

output: f'{cwd:a}/{annotation_name}.joint_tau.initial_processed_stats.rds' if joint_tau else f'{cwd:a}/{annotation_name}.single_tau.initial_processed_stats.rds'
task: trunk_workers = 1, trunk_size = job_size, walltime = walltime, mem = mem, cores = numThreads

R: expand= "${ }", stderr = f'{_output[0]}.stderr', stdout = f'{_output[0]}.stdout'
    library(data.table)
    print(paste("Joint tau analysis:", ${str(joint_tau).upper()}))
    is_joint <- ${"TRUE" if joint_tau else "FALSE"}
    print(paste("Is joint analysis:", is_joint))
    provided_num_non_annotation_cols = ${num_non_annotation_cols}   
    standard_non_annot_cols <- c("CHR", "SNP", "BP", "CM", "A1", "A2", "MAF")

    # Function to auto-calculate Mref from ldscore files
    calculate_mref <- function(target_anno_dir) {
        message("Auto-calculating Mref from ldscore files...")
        
        # Find ldscore files (both .gz and .parquet formats)
        ldscore_files <- list.files(target_anno_dir, 
                                  pattern = "\\.l2\\.ldscore\\.(gz|parquet)$", 
                                  full.names = TRUE)
        
        if (length(ldscore_files) == 0) {
            stop("No ldscore files found in directory: ", target_anno_dir)
        }
        
        message("Found ", length(ldscore_files), " ldscore files:")
        
        total_snps <- 0
        file_details <- data.frame(
            file = character(),
            format = character(), 
            rows = integer(),
            stringsAsFactors = FALSE
        )
        
        for (file in ldscore_files) {
            tryCatch({
                if (endsWith(file, ".parquet")) {
                    suppressPackageStartupMessages(library(arrow))
                    data <- arrow::read_parquet(file)
                    file_format <- "parquet"
                } else if (endsWith(file, ".gz")) {
                    data <- fread(file)
                    file_format <- "gz"
                } else {
                    warning("Unsupported file format: ", file)
                    next
                }
                
                n_rows <- nrow(data)
                total_snps <- total_snps + n_rows
                
                file_details <- rbind(file_details, data.frame(
                    file = basename(file),
                    format = file_format,
                    rows = n_rows
                ))
                
                message("  ", basename(file), " (", file_format, "): ", n_rows, " SNPs")
                
            }, error = function(e) {
                warning("Error reading file ", file, ": ", e$message)
            })
        }
        
        message("Total SNPs across all files: ", total_snps)
        message("Using Mref = ", total_snps)
        
        return(total_snps)
    }
    
    # Determine Mref value
    mref_param <- ${Mref}
    if (mref_param > 0) {
        Mref <- mref_param
        message("Using provided Mref value: ", Mref)
    } else {
        Mref <- calculate_mref("${target_anno_dir}")
    }

    calculate_sd_annot <- function(cell_path, annot_index = 1) {
    ll <- list.files(cell_path, pattern = "\\.annot\\.gz$", full.names = TRUE)
    if(length(ll) == 0) {
        stop("No annotation files found in: ", cell_path)
    }

    # detect number of non-annotation cols 
    if(provided_num_non_annotation_cols > 0) {
        num_non_annotation_cols <- provided_num_non_annotation_cols
        message("Using provided num_non_annotation_cols: ", num_non_annotation_cols)
    } else {
        # calculate automatically
        first_file <- ll[1]
        sample_dat <- fread(first_file, nrows = 5)
        file_cols <- names(sample_dat)
        present_standard_cols <- intersect(file_cols, standard_non_annot_cols)
        num_non_annotation_cols <- length(present_standard_cols)
        
        message("Auto-detected annotation file structure:")
        message("  Total columns: ", length(file_cols))
        message("  Standard non-annotation columns found: ", paste(present_standard_cols, collapse=", "))
        message("  Number of non-annotation columns: ", num_non_annotation_cols)
    }
    
    num = numeric(length(annot_index))
    den = 0
    
    for(m in ll) {
        dat <- fread(m, fill = TRUE)
        if(length(annot_index) > 1) {
            cols <- paste0("ANNOT_", annot_index) 
            num <- num + ((nrow(dat)-1) * sapply(dat[, ..cols], var))
        } else {
            if((num_non_annotation_cols + annot_index) > ncol(dat)) stop(paste("Index out of range:", m))
            num <- num + ((nrow(dat)-1) * var(dat[, num_non_annotation_cols+annot_index, with=FALSE]))
        }
        den <- den + (nrow(dat)-1)
    }
    sqrt(num/den)
    }

    check_file_exists <- function(file_path) {
        if (!file.exists(file_path)) {
            warning(paste("File does not exist:", file_path))
            return(FALSE)
        }
        return(TRUE)
    }

    # Process single trait
    process_trait <- function(trait, cwd) {
    files <- list(
        result = paste0(cwd, "/", trait, ".results"),
        log = paste0(cwd, "/", trait, ".log"),
        delete = paste0(cwd, "/", trait, ".part_delete")
    )
    
    if(!all(sapply(files, file.exists))) {
        warning(paste("Missing files for trait:", trait))
        return(NULL)
    }

    tryCatch({
        results <- fread(files$result)
        h2g <- as.numeric(gsub(".*h2: (-?[0-9.]+).*", "\\1",
                            grep("Total Observed scale h2:", readLines(files$log), value=TRUE)))
        delete_values <- fread(files$delete)
        
        list(trait = trait, h2g = h2g, results = results, delete_values = delete_values)
    }, error = function(e) {
        warning(paste("Error processing trait:", trait, "-", e$message))
        NULL
    })
    }

    # Process tau analysis
    process_tau_analysis <- function(trait_data, sd_annots, base_index = NULL, is_joint = FALSE, Mref) {
    
    if(is_joint) {
        indices <- 1:(nrow(trait_data$results) - base_index)
        sc_matrices <- matrix(0, nrow=nrow(trait_data$delete_values), ncol=length(indices))
        
        for(i in seq_along(indices)) {
            coef <- sd_annots[i] * Mref / trait_data$h2g
            sc_matrices[,i] <- trait_data$delete_values[[indices[i]]] * coef
        }
        
        list(joint_stats = list(
            h2g = trait_data$h2g,
            sd_annots = sd_annots,
            sc_matrices = sc_matrices
        ))
    } else {
        coef <- sd_annots * Mref / trait_data$h2g
        sc_matrix <- trait_data$delete_values[[1]] * coef
        
        list(single_tau = list(
            h2g = trait_data$h2g,
            sd_annot = sd_annots,
            sc_matrix = matrix(as.vector(trait_data$delete_values[[1]] * coef), ncol=1)
        ))        
    }
    }

    # Process enrichment
    process_enrichment <- function(trait_data, Mref) {
    enrichment_p <- as.numeric(trait_data$results[1, "Enrichment_p"])
    enrich_ratio <- ((trait_data$results[1, "Prop._h2"] / trait_data$results[1, "Prop._SNPs"]) - 
                        (1 - trait_data$results[1, "Prop._h2"]) / (1 - trait_data$results[1, "Prop._SNPs"]))
    enrich_term <- trait_data$h2g / Mref * enrich_ratio
    
    list(
        enrichment_summary = data.table(
            Enrichment = trait_data$results[1, "Enrichment"],
            Enrichment_std_error = trait_data$results[1, "Enrichment_std_error"],
            Prop._h2 = trait_data$results[1, "Prop._h2"],
            Prop._SNPs = trait_data$results[1, "Prop._SNPs"],
            Enrichment_p = enrichment_p
        ),
        meta_enrstat = list(
            enrich_stat = as.numeric(enrich_term),
            enrich_z = as.numeric(qnorm(enrichment_p / 2)),
            enrich_sd = as.numeric(enrich_term / qnorm(enrichment_p / 2))
        ),
        meta_enr = list(
            Enrichment = as.numeric(trait_data$results[1, "Enrichment"]),
            Enrichment_std_error = as.numeric(trait_data$results[1, "Enrichment_std_error"]) 
        )
    )
    }

    # Main analysis
    traits <- scan("${all_traits_file}", what = "character")
    results <- list()
    problematic_traits <- c()

    if (is_joint) {
    # Get baseline info and indices
    base_path <- paste0("${baseline_ld_dir}", "/", "${baseline_name}", "22.annot")
    if (file.exists(paste0(base_path, ".parquet"))) {
        base <- arrow::read_parquet(paste0(base_path, ".parquet"))
    } else if (file.exists(paste0(base_path, ".gz"))) {
        base <- fread(paste0(base_path, ".gz"))
    } else {
        stop("Neither .parquet nor .gz file found for baseline annotation")
    }    

    if(provided_num_non_annotation_cols > 0) {
        base_num_non_annotation_cols <- provided_num_non_annotation_cols
        message("Using provided num_non_annotation_cols for baseline: ", base_num_non_annotation_cols)
    } else {
        base_cols <- names(base)
        base_standard_cols <- intersect(base_cols, standard_non_annot_cols)
        base_num_non_annotation_cols <- length(base_standard_cols)
        message("Auto-detected baseline structure:")
        message("  Total columns: ", ncol(base))
        message("  Non-annotation columns: ", base_num_non_annotation_cols)
    }

    base_index <- ncol(base) - base_num_non_annotation_cols
    
    tab2 <- fread(paste0("${cwd}", "/", traits[1], ".results"))
    indices <- 1:(nrow(tab2) - base_index)
    
    sd_annots <- calculate_sd_annot("${target_anno_dir}", indices)
    
    for (trait in traits) {
        if (!is.null(trait_data <- process_trait(trait, "${cwd}"))) {
            results[[trait]] <- process_tau_analysis(trait_data, sd_annots, base_index, TRUE, Mref)
        } else {
            problematic_traits <- c(problematic_traits, trait)
        }
    }
    } else {
    sd_annot <- calculate_sd_annot("${target_anno_dir}")
    
    for (trait in traits) {
        if (!is.null(trait_data <- process_trait(trait, "${cwd}"))) {
            results[[trait]] <- c(
                process_tau_analysis(trait_data, sd_annot, NULL, FALSE, Mref),
                list(enrichment = process_enrichment(trait_data, Mref))
            )
        } else {
            problematic_traits <- c(problematic_traits, trait)
        }
    }
    }

    if (length(problematic_traits) > 0) {
    warning("Problematic traits:", paste(problematic_traits, collapse=", "))
    }

    saveRDS(results, "${_output[0]}", compress = "xz")

In [ ]:
[processed_stats_2]
parameter: trait_group_paths = []     
parameter: trait_group_names = []
input: group_by = "all" 
output: f'{cwd:a}/{step_name}/{"joint_tau" if joint_tau else "single_tau"}.{annotation_name}.meta_processed_stats.rds'

task: trunk_workers = 1, trunk_size = job_size, walltime = walltime, mem = mem, cores = numThreads

R: expand = '${ }', stderr = f'{_output[0]}.stderr', stdout = f'{_output[0]}.stdout'
  library(data.table)
  library(rmeta)

  processed_path = ${_input:ar}
  results_data <- readRDS(processed_path)
  is_joint <- ${str(joint_tau).upper()}
  
  process_paths <- function(input_str) {
      cleaned <- gsub("^\\[|\\]$", "", input_str)
      cleaned <- gsub("'", "", cleaned)
      paths <- strsplit(cleaned, ",\\s*|\\s+")[[1]]
      paths <- trimws(paths)
      return(paths)
  }

  group_paths <- process_paths("${trait_group_paths}")
  group_names <- process_paths("${trait_group_names}")
  problematic_traits <- list()
  joint_names <- process_paths("${joint_annotation_names}")
  if (is_joint) {
      joint_tau_results <- list()
      
      for (i in 1:length(group_paths)) {
          traits <- readLines(group_paths[i])
          group_stats <- list()
          
          for (trait in traits) {
              if(trait %in% names(results_data) && !is.null(results_data[[trait]]$joint_stats)) {
                  group_stats[[trait]] <- results_data[[trait]]$joint_stats
              } else {
                  warning(paste("Invalid data for trait:", trait))
                  problematic_traits[[trait]] <- "Invalid data"
              }
          }
          
          if(length(group_stats) > 0) {
              n_annotations <- ncol(group_stats[[1]]$sc_matrices)
              meta_result <- matrix(0, nrow = n_annotations, ncol = 3)
              colnames(meta_result) <- c("Mean", "SD", "P")
              if(length(joint_names) > 0) {
                  rownames(meta_result) <- joint_names  # Add row names
              }              
              for(j in 1:n_annotations) {
                  df <- matrix(0, nrow = length(group_stats), ncol = 2)
                  for(k in 1:length(group_stats)) {
                      sc_values <- group_stats[[k]]$sc_matrices[,j]
                      df[k,] <- c(mean(sc_values), sqrt((${n_blocks}-1)^2/${n_blocks} * var(sc_values)))
                  }
                  
                  meta <- meta.summaries(df[,1], df[,2], method = "random")
                  meta_result[j,] <- c(
                      meta$summary, 
                      meta$se.summary,
                      2 * pnorm(-abs(meta$summary/meta$se.summary))
                  )
              }
              joint_tau_results[[group_names[i]]] <- meta_result
          }
      }
      saveRDS(joint_tau_results, '${_output[0]}')
      
  } else {
      tau_results <- list()
      enrichment_results <- list()
      
      for (i in 1:length(group_paths)) {
          traits <- readLines(group_paths[i])
          
          group_stats <- Filter(function(x) {
              !is.null(x) && !is.null(x$single_tau$sc_matrix)
          }, results_data[traits])
          
          if (length(group_stats) > 0) {
              sc_data <- lapply(group_stats, function(x) {
                  mean_sc <- mean(x$single_tau$sc_matrix[, 1])
                  se_sc <- sqrt((${n_blocks}-1)^2 / ${n_blocks} * var(x$single_tau$sc_matrix[, 1]))
                  return(c(mean_sc, se_sc))
              })
              sc_matrix <- do.call(rbind, sc_data)
              
              meta_tau <- meta.summaries(sc_matrix[, 1], sc_matrix[, 2], method = "random")
              tau_results[[group_names[i]]] <- c(meta_tau$summary, 
                                              meta_tau$se.summary,
                                              2 * pnorm(-abs(meta_tau$summary / meta_tau$se.summary)))
              
              enr_data <- lapply(group_stats, function(x) {
                  return(c(x$enrichment$meta_enr$Enrichment,
                          x$enrichment$meta_enr$Enrichment_std_error))
              })
         
              enr_matrix <- do.call(rbind, enr_data)
              valid_rows <- !is.na(enr_matrix[,1]) & !is.na(enr_matrix[,2]) & 
                          !is.nan(enr_matrix[,1]) & !is.nan(enr_matrix[,2]) &
                          enr_matrix[,2] > 0               
              trait_names <- names(group_stats)
              invalid_traits <- trait_names[!valid_rows]

              message(sprintf("Group %s - Total: %d, Valid: %d, Invalid: %d", 
                              group_names[i], nrow(enr_matrix), sum(valid_rows), sum(!valid_rows))) 
              if(sum(!valid_rows) > 0) {
                  invalid_traits <- trait_names[!valid_rows]
                  invalid_se_values <- enr_matrix[!valid_rows, 2]
                
                  message(sprintf("Invalid traits in group %s (SE = 0):", group_names[i]))
                  for(j in 1:length(invalid_traits)) {
                      se_val <- invalid_se_values[j]
                      if(is.na(se_val) || is.nan(se_val)) {
                          message(sprintf("  - %s: SE = NA/NaN", invalid_traits[j]))
                      } else if(se_val == 0) {
                          message(sprintf("  - %s: SE = 0", invalid_traits[j]))
                      } 
                  }
              }                                                       
              enrstat_data <- lapply(group_stats, function(x) {
                  return(c(x$enrichment$meta_enrstat$enrich_stat, x$enrichment$meta_enrstat$enrich_sd))})
              enrstat_matrix <- do.call(rbind, enrstat_data)

              if(sum(valid_rows) >= 2) {
                  meta_enr <- meta.summaries(enr_matrix[valid_rows, 1], 
                                          enr_matrix[valid_rows, 2], 
                                          method = "random")
              } else {
                  warning(paste("Insufficient valid data for", group_names[i]))
                  meta_enr <- list(summary = NA, se.summary = NA)
              }              
              meta_enrstat <- meta.summaries(enrstat_matrix[, 1], enrstat_matrix[, 2], method = "random")   
              enrichment_results[[group_names[i]]] <- c(meta_enr$summary,
                                                      meta_enr$se.summary,
                                                       2 * pnorm(-abs(meta_enrstat$summary / meta_enrstat$se.summary)))
          } else {
              warning(paste("No valid data for group:", group_names[i]))
          }
      }


                
      tau_df <- do.call(rbind, tau_results)
      enrichment_df <- do.call(rbind, enrichment_results)
      
      colnames(tau_df) <- c("Mean", "SD", "P")
      colnames(enrichment_df) <- c("Mean", "SD", "P")
      rownames(tau_df) <- group_names
      rownames(enrichment_df) <- group_names
      
      results <- list(tau = tau_df, enrichment = enrichment_df)
      saveRDS(results, '${_output[0]}')
  }

  if (length(problematic_traits) > 0) {
      warning("The following traits had issues:")
      print(problematic_traits)
  }